# Рейтинг горнолыжных курортов России

**Заказчик:** Travel Designer

**Исполнитель:** Оксана Блюхерова

**Дата:** ноябрь 2026

## Цели и задачи

**Цель:** Выбрать горнолыжные курорты, которые стоит добавить в каталог Travel Designer в первую очередь, при ограниченных ресурсах на подключение.

**Задачи:**
1. Собрать данные из нескольких источников по факторам, которые могут влиять на рейтинг курорта.
2. Проверить качество и полноту данных, подготовить данные из каждого источника к дальнейшему анализу.
3. Построить составной рейтинг по каждому из курортов (одна цифра рейтинга для одного курорта), а также дать оценку степени уверенности в данных.
4. Проверить устойчивость рейтинга при разных сценариях оценки факторов.
5. Сформулировать пояснения, какие факторы и как повлияли на итоговый рейтинг курорта.
6. Сформулировать выводы и подготовить рекомендации.

In [1]:
# Импортируем библитеки
import json
import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)

In [2]:
# Загрузка полученных данных
sources = pd.read_csv('../raw/sources.csv')
resorts_data = pd.read_csv('../mart/resort_features.csv')
resort_scores = pd.read_csv('../mart/resort_scores.csv')
contribution_scores = pd.read_csv('../mart/contribution_scores.csv')


Все таблицы, которые используются в ноутбуке, собраны пайплайном из src/ и пересобираются одной командой python src/run_pipeline.py (подробности — в README). Ноутбук только читает готовые файлы из mart/ и ничего не пересчитывает.

## Исходные данные и ограничения

In [3]:
print('Для анализа и ранжирования использованы следующие источники')
display(sources[['source_name', 'method', 'url', 'collected_at', 'data_period', 'transformations', 'notes']])

Для анализа и ранжирования использованы следующие источники


,source_name,method,url,collected_at,data_period,transformations,notes
0,Справочник курортов (материалы проекта),project_materials,NaN,2026-09-14,снимок на 2026-09-14,"нет, файл скопирован без изменений","Дата сбора в формате Excel, погодные условия и..."
1,Open-Meteo Historical Weather API,own_api,https://archive-api.open-meteo.com/v1/archive,2026-10-20,2021-11-01 - 2026-04-30,"нет, ответ API сохранён без изменений в конвер...","ERA5 вместо ERA5 Land из ТЗ, так как в модели ..."
2,Поездки по регионам Росстат,project_materials,https://www.fedstat.ru/indicator/62157,2026-09-28,2022-2025,"нет, файл скопирован без изменений","Данные собраны по месяцам, накопительно, для р..."
3,Ночёвки по регионам Росстат,project_materials,https://www.fedstat.ru/indicator/62489,2026-09-28,2023-2025 годы,"нет, файл скопирован без изменений","Данные собраны по кварталам, накопительно, для..."
4,Цены взрослого дневного скипаса,project_materials,NaN,2026-09-14,2025-12 - 2026-06,"нет, файл скопирован без изменений",Дата сбора в формате Excel. Цены есть для 9 из...
5,Проверка координат курортов (Яндекс Карты),manual,https://yandex.com/maps/,2026-10-30,снимок на 2026-10-30,исходные данные,Для сверки использовано единое правило: точка ...


Список курортов определен заранее. Все источники сведены в одну витрину resort_features: одна строка — один курорт, всего 32 курорта. Данные регионального спроса присоединены к курортам по региону, поэтому у курортов одного региона они одинаковые (например, Роза Хутор и Газпром Поляна).

Погода собрана самостоятельно по API, использована модель ERA5, вместо рекомендованной в ТЗ ERA5-Land, так как рекомендованная модель не отдавала данные по снегопадам.

Остальные данные взяты из материалов проекта и проверены. 

Дальнейшие расчеты производились на основании данных, полученных по координатам наставника. Рекомендовано дополнительно провести сбор данных по актуальным координатам (у 10 из 32 курортов точка справочника смещена от склона более чем на 3 км, максимум – 37 км у Евразии).

### Логика расчета и оценки популярности курортов (кратко)
Для каждого курорта будет выведена одна итоговая оценка (`popularity_score`) в каждом сценарии весов. Итоговый `popularity_score` рассчитывается по следующей схеме:
1. Для каждого курорта собраны 13 показателей в пяти группах: спрос в регионе, инфраструктура, зимние условия, доступность, цена.
2. Каждый курорт получает по большинству параметров промежуточный балл (от 0 до 100), который показывает положение курорта по отношению к остальным курортам. Исключения – рост поездок и средняя длительность поездки: для них шкала фиксированная.
3. Каждый промежуточный балл каждого курорта умножается на вес параметра в сценарии (сумма весов равна 100). Вес отражает важность параметра.
4. Произведения баллов и весов по 13 параметрам складываются и делятся на 100 – это и есть итоговая оценка `popularity_score` (от 0 до 100).
5. На основании рассчитанных оценок рассчитывается рейтинг: курорт с наивысшей оценкой получает ранг 1 и так далее. Курорты с одинаковой оценкой получают одинаковый ранг.
6. Дополнительно для каждого курорта на основании качества исходных данных рассчитывается степень уверенности в данных (например, отсутствие цены или неточные координаты снижают уверенность в итоговом балле). Уверенность не влияет на оценку и рейтинг, а показывает, каким местам рейтинга можно доверять больше, а каким меньше.

Результаты проверок – в следующем блоке.

## Проверка полноты и качества данных

Перед составлением сводной таблицы по курортам и их характеристикам выполняется проверка: 
- 32 строки;
- resort_id уникальный;
- нет пропусков спроса после соединения. 

Все три проверки пройдены: в витрине все 32 курорта без дублей, и у каждого есть данные регионального спроса.

Далее проверяем расчетные данные по тринадцати параметрам, на основании которых будет производиться расчет рейтинга.

In [4]:
# Проверка характеристик столбцов с расчетными параметрами
important_params = ['trips_2025', 'trips_growth_2025', 'avg_stay_nights_2025', 'piste_segments_6km',
                    'lift_lines_6km', 'lodging_objects_6km', 'food_attraction_objects_6km',
                    'avg_snow_days_per_season', 'avg_freezing_days_per_season', 'avg_snowfall_cm_per_season',
                    'nearest_airport_distance_km', 'transport_objects_6km', 'adult_skipass_avg_rub']
display(resorts_data[important_params].describe().T)

,count,mean,std,min,25%,50%,75%,max
trips_2025,32.00,"3,726,458.22","6,250,391.89","197,708.00","662,210.00","1,377,324.50","1,908,935.00","22,431,311.00"
trips_growth_2025,32.00,0.01,0.27,-0.36,-0.14,-0.03,0.05,0.86
avg_stay_nights_2025,32.00,3.92,1.55,1.75,2.89,3.56,4.43,7.83
piste_segments_6km,32.00,44.75,71.94,0.00,8.75,19.50,42.25,334.00
lift_lines_6km,32.00,12.22,15.84,0.00,3.50,8.00,13.00,64.00
lodging_objects_6km,32.00,53.03,72.34,0.00,5.50,18.00,64.00,255.00
food_attraction_objects_6km,32.00,71.28,83.00,0.00,21.25,44.50,80.50,341.00
avg_snow_days_per_season,32.00,70.58,21.79,17.00,62.05,71.70,82.20,107.80
avg_freezing_days_per_season,32.00,123.76,38.97,16.40,116.70,133.90,145.50,172.20
avg_snowfall_cm_per_season,32.00,152.72,78.64,31.29,88.97,144.05,199.85,321.36


Стоит обратить внимание, что не для всех курортов есть средняя цена взрослого skipass (колонка `adult_skipass_avg_rub`). Заполнены данные только для 9 из 32 курортов. Для отсутствующих значений будет использован балл 50 (нейтральная оценка по шкале от 0 до 100), чтобы отсутствие цены не поднимало и не опускало курорт; такие курорты получают флаг.

Для параметров цены (`adult_skipass_avg_rub`) и удаленность аэропорта (`nearest_airport_distance_km`) использована обратная логарифмическая шкала: чем ниже значение, тем выше балл.

Количество поездок в регионе курорта (`trips_2025`) варьируется достаточно сильно от 197,7 тыс. до 22,4 млн за год (максимум больше минимума более чем в 100 раз). Если использовать стандартное сравнение от минимума к максимуму, то распределение будет неравномерным и курорт региона с показателем 22,4 млн получит балл 100, а большинство оставшихся получат баллы около 0. Для корректного отражения положения каждого курорта использована логарифмическая нормализация, которая сравнивает «во сколько раз», а не «на сколько», тем самым делет распрделение баллов более равномерным.

Параметры по объектам инфраструктуры OSM (трассы `piste_segments_6km`, подъёмники `lift_lines_6km`, жильe `lodging_objects_6km`, питание и места интереса `food_attraction_objects_6km`, транспорт `transport_objects_6km`) также имеют сильный разброс значений от 0 до сотен, распределение сильно скошено, также используется логарифмическая нормализация. Аналогично и данные о снегопадах `avg_snowfall_cm_per_season` разброс данных от 31 и до 321 см, использована логарифмическая шкала.

Параметр роста поездок (`trips_growth_2025`) рассчитан как относительное изменение числа поездок в 2025 году по сравнению с 2024 годом: поездки 2025 / поездки 2024 − 1. Значение 0,12 означает рост на 12%, отрицательное значение – падение. В данных есть сильные отклонения: прирост +86% (Карачаево-Черкесия) и падение −36% (Северная Осетия). Чтобы такие выбросы не искажали оценку, использована линейная шкала с фиксированными границами. Внутри +/-25% балл меняется линейно: 0% — 50 баллов, +10% — 70, −10% — 30. Рост на 25% и больше даёт 100 баллов, падение на 25% и больше — 0 баллов.

Параметр среднее количество ночей за поездку (`avg_stay_nights_2025`) варьируется от 1,75 до 7,83. Для этого параметра также использована фиксированная шкала: одна ночь дает 0 баллов, семь ночей и больше - 100 баллов. Между ними балл растёт линейно: например, 4 ночи — 50 баллов

Значения показателей снежные `avg_snow_days_per_season` и морозные дни `avg_freezing_days_per_season` распределены равномернее, без сильного скоса, поэтому использована обычная шкала min–maxx. 

Важно отметить, что у части курортов в радиусе 6 км не найдено ни одной трассы или подъёмника, а у некоторых нет объектов питания, размещения или транспорта. Для действующих горнолыжных курортов это маловероятно, поэтому нули, скорее всего, связаны с качеством данных: объекты OSM искались вокруг точки справочника, а она может не совпадать с реальным положением склона. Чтобы это проверить, сопоставим курорты с нулевыми значениями с результатами сверки координат.

In [5]:
# Выгрузим сверку координат по курортам с 0 значениями по объектам инфраструктуры
osm_columns = ['piste_segments_6km', 'lift_lines_6km', 'lodging_objects_6km',
              'food_attraction_objects_6km', 'transport_objects_6km']
haz_zero = (resorts_data[osm_columns] == 0).any(axis=1)
resorts_without_infrastructure = resorts_data[haz_zero][['name_ru', 'region', 'distance_km', 'coordinate_check'] + osm_columns]
display(resorts_without_infrastructure)


,name_ru,region,distance_km,coordinate_check,piste_segments_6km,lift_lines_6km,lodging_objects_6km,food_attraction_objects_6km,transport_objects_6km
8,Мамисон,Республика Северная Осетия — Алания,0.89,exact,12,2,0,10,0
16,Холдоми,Хабаровский край,11.39,mismatch,0,0,0,0,3
17,Морозная гора,Камчатский край,11.12,mismatch,0,0,18,39,148
24,Завьялиха,Челябинская область,5.74,mismatch,5,2,1,3,0
31,Гладенькая,Республика Хакасия,17.42,mismatch,0,0,3,10,84


У Мамисона координаты точные (смещение 0,9 км), поэтому отсутствие объектов размещения в радиусе 6 км может отражать реальность: курорт новый и удалённый. 

У Холдоми, Морозной горы и Гладенькой точка справочника смещена от склона на 11–17 км. Склон не попадает в радиус поиска, поэтому трассы и подъёмники не найдены. Нули здесь, скорее всего, вызваны координатами, а не отсутствием инфраструктуры.

У Завьялихи смещение 5,7 км: часть трасс и подъёмников найдена, но объекты, вероятно, учтены не полностью.

Неточные координаты влияют и на сам рейтинг (через заниженные баллы инфраструктуры), и на уверенность: эти курорты не получают баллов уверенности за точные координаты. Но формула уверенности не различает смещение на 1 км и на 17 км, поэтому эти четыре курорта отдельно отмечены в выводах.
____
Стоит также выделить ряд курортов, у которых могут пересекаться объекты OSM, так как курорты расположены рядом.

In [6]:
# Отобразим курорты, у которых пересекаются объекты OSM
resorts_overlap_osm = resorts_data[resorts_data['osm_overlap_warning'].notna()][['name_ru','region', 'osm_overlap_warning']]
display(resorts_overlap_osm)

,name_ru,region,osm_overlap_warning
0,Роза Хутор,Краснодарский край,"Possible overlap with Газпром Поляна (6.0 km),..."
1,Красная Поляна,Краснодарский край,"Possible overlap with Газпром Поляна (1.6 km),..."
2,Газпром Поляна,Краснодарский край,"Possible overlap with Красная Поляна (1.6 km),..."
6,Эльбрус,Кабардино-Балкарская Республика,Possible overlap with Чегет (1.4 km)
7,Чегет,Кабардино-Балкарская Республика,Possible overlap with Эльбрус (1.4 km)
8,Мамисон,Республика Северная Осетия — Алания,Possible overlap with Цей (11.5 km)
9,Цей,Республика Северная Осетия — Алания,Possible overlap with Мамисон (11.5 km)
12,Большой Вудъявр,Мурманская область,"Possible overlap with КолаСпортЛэнд (1.7 km), ..."
13,Кукисвумчорр,Мурманская область,Possible overlap with Большой Вудъявр (6.7 km)...
14,КолаСпортЛэнд,Мурманская область,Possible overlap with Большой Вудъявр (1.7 km)...


Для курортов, расположенных близко друг к другу, радиусы поиска объектов OSM (6 км) пересекаются, и часть объектов учтена у нескольких курортов сразу. Для действительно соседних курортов это отражает реальность: например, гости курортов Красной Поляны могут пользоваться общей инфраструктурой долины. Но для рейтинга это значит, что по инфраструктуре такие курорты получают почти одинаковые баллы и различаются в основном другими показателями.
____
Дополнительно проверим качество координат по всем курортам.

In [7]:
# Выведем количество курортов по группам качества координат
coordinat_quality_groups = resorts_data.groupby('coordinate_check')['resort_id'].count()
print('Количество курортов в каждой из групп по кажеству координат:')
display(coordinat_quality_groups)

Количество курортов в каждой из групп по кажеству координат:


coordinate_check
approximate     8
exact          14
mismatch       10
Name: resort_id, dtype: int64

Точные координаты (смещение до 1 км) у 14 курортов из 32. Для остальных 18 отразим, насколько точка справочника, от которой собирались данные об инфраструктуре и погоде, удалена от реального положения склона.

In [8]:
# Выведем список курортов с неточными координатами
missed_locations = resorts_data[resorts_data['coordinate_check'] != 'exact'][['name_ru', 'region', 'distance_km', 'coordinate_quality']]
display(missed_locations.sort_values(by='distance_km', ascending=False).reset_index(drop=True))

,name_ru,region,distance_km,coordinate_quality
0,Евразия,Челябинская область,37.16,manual_review_required
1,Гладенькая,Республика Хакасия,17.42,nearby_feature_proxy
2,Холдоми,Хабаровский край,11.39,manual_review_required
3,Морозная гора,Камчатский край,11.12,manual_review_required
4,Завьялиха,Челябинская область,5.74,manual_review_required
5,Мраткино,Республика Башкортостан,5.33,manual_review_required
6,Телецкий,Республика Алтай,5.18,manual_review_required
7,Аджигардак,Челябинская область,4.16,manual_review_required
8,Роза Хутор,Краснодарский край,3.94,nearby_feature_proxy
9,Эльбрус,Кабардино-Балкарская Республика,3.10,nearby_feature_proxy


У 10 из 32 курортов точка справочника смещена от склона больше чем на 3 км. 8 из них в справочнике уже помечены как требующие ручной проверки, ещё 2 — как точки ближайшего объекта: проверка подтвердила эти предупреждения и дала им оценку в километрах. Для пяти курортов установлено, что точка стоит в населённом пункте: Терскол (Эльбрус), Артыбаш (Телецкий), Белорецк (Мраткино), Трёхгорный (Завьялиха), Саяногорск (Гладенькая). От этих точек считались погода, объекты OSM и расстояние до аэропорта, поэтому чем больше смещение, тем менее надёжны показатели курорта.
____
Погода собрана самостоятельно (модель ERA5 из Open-Meteo), а в справочнике проекта есть свои значения тех же показателей. Источник значений справочника неизвестен: в Open-Meteo модель ERA5-Land, рекомендованная в ТЗ, снегопад не отдаёт. Сравним наши значения со значениями справочника, чтобы понять, насколько можно доверять погодным показателям.

In [9]:
# Сравним погодные данные в разных моделях
with open('../raw/resorts.json', 'r', encoding='utf-8') as f:
        resorts = json.load(f)
        era5_land_weather_df = pd.DataFrame(resorts[1:], columns=resorts[0])
weather_check = era5_land_weather_df.merge(
        resorts_data[['resort_id', 'avg_snowfall_cm_per_season', 'avg_snow_days_per_season','avg_freezing_days_per_season']],
        on='resort_id',
        how='left',
        suffixes=('_era5_land','_era5')
)
weather_check['snowfall_diff'] = weather_check['avg_snowfall_cm_per_season_era5'] - weather_check['avg_snowfall_cm_per_season_era5_land']
weather_check['snow_day_diff'] = weather_check['avg_snow_days_per_season_era5'] - weather_check['avg_snow_days_per_season_era5_land']
weather_check['friz_days_diff'] = weather_check['avg_freezing_days_per_season_era5'] - weather_check['avg_freezing_days_per_season_era5_land']

display(weather_check[['name_ru', 'snowfall_diff', 'snow_day_diff', 'friz_days_diff']].sort_values('snowfall_diff').reset_index(drop=True))

,name_ru,snowfall_diff,snow_day_diff,friz_days_diff
0,Красная Поляна,-358.79,-75.60,-0.40
1,Газпром Поляна,-353.84,-75.40,-0.20
2,Роза Хутор,-269.74,-59.60,0.80
3,Архыз,-263.97,-44.40,-0.60
4,Домбай,-242.96,-42.40,0.60
5,Морозная гора,-221.84,-24.00,-2.00
6,Эльбрус,-119.99,-17.80,0.20
7,Чегет,-118.30,-17.80,0.00
8,Мамисон,-98.25,-26.60,0.20
9,Телецкий,-64.76,-11.60,1.20


Данные из Open-Meteo (модель ERA5) и данные справочника почти совпадают по числу морозных дней (расхождение в пределах +/-2 дней), но сильно расходятся по снегу. Значения снегопада и снежных дней из Open-Meteo ниже. Сильнее всего расхождение у Красной Поляны и Газпром Поляны: снегопада меньше примерно на 355–359 см за сезон, снежных дней — примерно на 75. У части курортов значения близки. Источник значений справочника точно неизвестен, поэтому проверить, какая оценка ближе к реальности, нельзя. 

Вывод: снежные показатели некоторых курортов в собранных данных могут быть занижены. В климатическом сценарии, где вес снега выше, эти курорты могут оказаться ниже, чем на самом деле.

____
Перейдем к проверке качества и полноты данных по регионам. 

В пайплайне предобработаны сырые данные из двух источников: «Поездки по регионам Росстат» и «Ночёвки по регионам Росстат». Выбраны данные по поездкам за 2024 и 2025 годы и количеству ночевок за 2025 год по каждому региону. Рассчитаны необходимые параметры: поездки по региону за 2025 год, среднее количество ночевок за поездку и рост количества поездок в 2025 году по сравнению с 2024 годом. Для корректного добавления данных к курортам проведена сверка названий регионов. Для двух регионов (список ниже) названия сопоставлены через словарь.

In [10]:
# Регионы с несовпадающими названиями
unmatched_region_names = resorts_data[resorts_data['region'] != resorts_data['region_rosstat']][['name_ru', 'region', 'region_rosstat']]
display(unmatched_region_names)

,name_ru,region,region_rosstat
3,Шерегеш,Кемеровская область,Кемеровская область - Кузбасс
8,Мамисон,Республика Северная Осетия — Алания,Республика Северная Осетия - Алания
9,Цей,Республика Северная Осетия — Алания,Республика Северная Осетия - Алания


Проверим для каких регионов наблюдался аномальный рост или падение поездок.

In [11]:
# Регионы с аномальными изменениями поездок
exceeding_region_trips_growth = resorts_data[abs(resorts_data['trips_growth_2025']) >= 0.25][['region', 'trips_growth_2025']].drop_duplicates()
display(exceeding_region_trips_growth.sort_values('trips_growth_2025', ascending=False).reset_index(drop=True))

,region,trips_growth_2025
0,Карачаево-Черкесская Республика,0.86
1,Камчатский край,0.26
2,Республика Хакасия,0.25
3,Республика Северная Осетия — Алания,-0.36


У четырёх регионов рост поездок за год выходит за +/-25%: Карачаево-Черкесия (+86%), Камчатка (+26%), Хакасия (+25%), Северная Осетия (−36%). Рост посчитан по одному году (2025 к 2024), поэтому резкие изменения не стоит читать как сигнал об однозначном изменении привлекательности курортов. Они могут быть вызваны низкой базой 2024 года, разовыми событиями или изменениями учёта. Поэтому балл роста ограничен шкалой +/-25%, и выброс не может перевесить другие показатели.

Числа для трёх регионов (Карачаево-Черкесия, Северная Осетия, Краснодарский край) вручную сверены с исходным файлом ЕМИСС и пересчитаны – расчёт верный. Для Карачаево-Черкесии сильный рост в 2025 году подтверждают и открытые источники.

Чтобы оценить устойчивость роста, его стоит смотреть за несколько лет — это следующий шаг.

Поездки учитываются за весь год, поэтому в них входят и туристы, которые приезжают в регион не ради горных лыж: например, в Краснодарский край едут и на море. Спрос в регионе — косвенная оценка спроса на курорт.
____
Првоерим количество курортов по регионам.

In [12]:
# Проверим количество курортов по регионам
resorts_count_by_region = resorts_data['region'].value_counts()
display(resorts_count_by_region)

region
Челябинская область                    4
Краснодарский край                     3
Мурманская область                     3
Карачаево-Черкесская Республика        2
Кабардино-Балкарская Республика        2
Республика Северная Осетия — Алания    2
Республика Алтай                       2
Республика Башкортостан                2
Самарская область                      1
Московская область                     1
Ленинградская область                  1
Пермский край                          1
Красноярский край                      1
Свердловская область                   1
Иркутская область                      1
Кемеровская область                    1
Камчатский край                        1
Хабаровский край                       1
Сахалинская область                    1
Республика Хакасия                     1
Name: count, dtype: int64

В 8 регионах находи
тся от 2 до 4 курортов, вместе это 20 из 32 курортов. Курорты одного региона получают одинаковые показатели спроса (поездки, рост, длительность), а значит, и одинаковые баллы по этой группе. Различаются они только инфраструктурой, зимними условиями, доступностью и ценой. Спрос весит 40% индекса, поэтому это заметное ограничение модели: спрос на конкретный курорт может сильно отличаться от спроса на регион в целом.
___
В процессе проверки обнаружено, что 2 курорта закрыты - список ниже. Проверка по карточкам компаний в Яндекс картах.

In [13]:
# Закрытые курорты
closed_resorts = resorts_data[resorts_data['is_closed']][['resort_id', 'name_ru']]
display(closed_resorts)

,resort_id,name_ru
14,R015,КолаСпортЛэнд
22,R023,Аджигардак


### Выводы по разделу

После анализа полноты и качества данных можно сформулировать следующие выводы и ограничения.
1. Цены указаны только для 9 из 32 курортов, остальные 23 курорта получают нейтральный балл.
2. Для 10 курортов координаты отличаются от координат, по которым получены данные об инфраструктуре и погоде, более чем на 3 км (максимальное отклонение – 37 км), что может искажать итоговый рейтинг. Например, у Холдоми, Морозной горы и Гладенькой вероятно из-за этого не найдена инфраструктура, и их места занижены.
3. Климатические данные собраны самостоятельно из источника OpenMeteo (модель ERA5) и они отличаются для некоторых курортов, по сравнению со справочником. Снежные показатели части курортов, вероятно, занижены, это влияет на климатический сценарий.
4. У четырёх регионов рост поездок за год выходит за +/-25%. Чтобы выбросы не искажали рейтинг, использована фиксированная шкала: за этими пределами балл не растёт, и резкий рост не трактуется как сигнал привлекательности.
5. Названия двух регионов сопоставлены словарем для корректной привязки курортов к регионам, все курорты получили данные спроса.
6. Спрос задан на уровне региона: у 20 курортов он общий с соседями, спрос не разделяется по типам отдыха, поэтому может содержать спрос по туризму с другими целями (например пляжный отдых).

Дополнительно: два курорта «КолаСпортЛэнд» и «Аджигардак» закрыты. Из расчета пока не исключены. Они не вошли в ТОП рекомендаций.

## Методика расчета и веса параметров 

Для оценки устойчивости ТОПа курортов выбрано 3 сценария:
1. Базовый сценарий - первоначальная гипотеза заказчика. Ранжирование по этому сценарию можно расценивать как наиболее приоритетный порядок добавления курортов на платформу.
2. Климатический сценарий - увеличение веса на 20 пунктов для группы параметров связанных с зимними условиями при пропорциональном снижении веса остальных параметров.
3. Сценарий доступности - больше веса отдается на объекты категории доступность.

Сдвиг веса для приоритетной группы параметров в сценарии +20 пунктов выбран как заметный, но не доминирующий. При другой величине места изменятся иначе. 

Распределение весов по группам и параметрам представлено ниже. Сумма весов в каждом сценарии равна 100. Проверка на выполнение этого условия заложена в пайплайн.

In [14]:
# Импорт распределения весов парамеров по сценариям
parameters_weights_by_scenario = pd.read_csv('../mart/scoring_config.csv')
display(parameters_weights_by_scenario)

,parameter,column,param_group,basic_scenario,climate_scenario,access_scenario,demand_group_scenario,infrastructure_group_scenario,weather_group_scenario,accessibility_group_scenario,price_group_scenario
0,Поездки по региону,trips_2025,Региональный спрос,25,18.75,19.44,30.00,23.33,23.75,24.44,24.74
1,Рост поездок,trips_growth_2025,Региональный спрос,10,7.50,7.78,12.00,9.33,9.50,9.78,9.89
2,Средняя длительность,avg_stay_nights_2025,Региональный спрос,5,3.75,3.89,6.00,4.67,4.75,4.89,4.95
3,Трассы OSM,piste_segments_6km,Инфраструктура,10,7.50,7.78,8.67,12.00,9.50,9.78,9.89
4,Подъёмники OSM,lift_lines_6km,Инфраструктура,7,5.25,5.44,6.07,8.40,6.65,6.84,6.93
5,Размещение OSM,lodging_objects_6km,Инфраструктура,5,3.75,3.89,4.33,6.00,4.75,4.89,4.95
6,Питание и места интереса,food_attraction_objects_6km,Инфраструктура,3,2.25,2.33,2.60,3.60,2.85,2.93,2.97
7,Снежные дни,avg_snow_days_per_season,Зимние условия,8,16.00,6.22,6.93,7.47,9.60,7.82,7.92
8,Морозные дни,avg_freezing_days_per_season,Зимние условия,6,12.00,4.67,5.20,5.60,7.20,5.87,5.94
9,Сезонный снегопад,avg_snowfall_cm_per_season,Зимние условия,6,12.00,4.67,5.20,5.60,7.20,5.87,5.94


Точно неизвестно, что для аудитории важнее: спрос, снег или близость аэропорта, при этом разным путешественникам важны разные параметры. Базовые веса — гипотеза заказчика. Курорты, которые остаются в верхней части рейтинга во всех трёх сценариях, — наиболее надёжный выбор для старта каталога: они подходят достаточно широкой аудитории (исходя из рассотренных трех разных сценариев).

## Рейтинг курортов 

Рейтинг ТОП-10 курортов по базовому сценарию

In [15]:
# Выведем ТОП-10 курортов и их основные показатели
needed_columns = ['name_ru', 'region', 'popularity_score_basic_scenario', 'confidence_score', 'is_closed','quality_flags']
display(resort_scores[needed_columns].head(10))

,name_ru,region,popularity_score_basic_scenario,confidence_score,is_closed,quality_flags
0,Сорочаны,Московская область,74.20,75.00,False,нет данных adult_skipass_avg_rub
1,Роза Хутор,Краснодарский край,66.05,75.00,False,coordinates are proxy; OSM radii overlap
2,Газпром Поляна,Краснодарский край,63.75,100.00,False,OSM radii overlap
3,Красная Поляна,Краснодарский край,61.25,100.00,False,OSM radii overlap
4,Игора,Ленинградская область,59.13,50.00,False,coordinates are proxy; нет данных adult_skipas...
5,Шерегеш,Кемеровская область,58.64,75.00,False,нет данных adult_skipass_avg_rub
6,Горный воздух,Сахалинская область,56.02,45.45,False,check coordinates; нет данных adult_skipass_av...
7,Архыз,Карачаево-Черкесская Республика,55.25,100.00,False,NaN
8,Домбай,Карачаево-Черкесская Республика,53.94,75.00,False,нет данных adult_skipass_avg_rub
9,Бобровый лог,Красноярский край,52.98,45.45,False,check coordinates; нет данных adult_skipass_av...


Первое место занимает «Сорочаны» (Московская область) с отрывом 8,15 балла от второго места. Уверенность — 75 из 100: для курорта нет данных о цене ски-пасса.

Следующие три места занимают курорты Краснодарского края: «Роза Хутор», «Газпром Поляна» и «Красная Поляна». Разница между соседними местами небольшая, 2–2,5 балла. У «Газпром Поляны» и «Красной Поляны» уверенность 100: все данные есть, координаты точные. У «Розы Хутор» — 75: не хватило точности координат. Все три курорта расположены очень близко друг к другу, их радиусы поиска объектов OSM пересекаются, поэтому часть объектов инфраструктуры учтена у нескольких курортов сразу, и их оценки по инфраструктуре похожи.

На 8-м и 9-м местах — два курорта Карачаево-Черкесской Республики: «Архыз» и «Домбай», с разницей 1,31 балла.

В базовом сценарии наибольший вес у спроса, а спрос задаётся на уровне региона и одинаков для всех курортов региона. Для Краснодарского края, где курорты расположены рядом, это допущение выглядит разумным. Но «Архыз» и «Домбай» находятся примерно в 45 км друг от друга, а спрос и балл по нему у них одинаковые, хотя реальный спрос на эти курорты может различаться.

У 6 из 10 курортов топ-10 нет данных о цене ски-пасса. Поэтому при добавлении всех цен рейтинг может немного измениться (на величину до 2,5 балла у каждого курорта без цены).

У большинства курортов топ-10 уверенность от 75 до 100. В топ-10 высокий индекс сочетается с низкой уверенностью у трёх курортов: «Игора» (5-е место, уверенность 50), «Горный воздух» (7-е, 45,45) и «Бобровый лог» (10-е, 45,45). У всех трёх нет данных о цене, а в справочнике их координаты помечены как неточные. Однако наша сверка показала, что точки близки к реальному положению курортов (смещение 0,1–2 км). Значит, их места надёжнее, чем показывает уверенность: формула ТЗ использует оценку координат из справочника и не учитывает результаты сверки. Главный источник неопределённости у этих курортов — отсутствие цены, а она влияет на индекс не более чем на 2,5 балла.
______
Проверим какие факторы сильнее всего повлияли на оценки курортов.

In [16]:
# Проверим, какие факторы сильне всего повлияли на оценки
columns = [col for col in contribution_scores.columns if col.endswith('_contribution_basic_scenario')]
parameters_contribution = contribution_scores[columns].std()
parameters_corr = contribution_scores[columns].corrwith(contribution_scores['popularity_score_basic_scenario'])
contribution = pd.DataFrame({
    'std': parameters_contribution,
    'corr': parameters_corr
}).sort_values('std', ascending=False)
display(contribution.style.background_gradient(subset=['std'], cmap='Blues').background_gradient(subset=['corr'], cmap='RdBu', vmin=-1, vmax=1))

,std,corr
trips_2025_contribution_basic_scenario,6.707512,0.741988
trips_growth_2025_contribution_basic_scenario,3.074159,0.421843
piste_segments_6km_contribution_basic_scenario,2.477224,0.687579
avg_snow_days_per_season_contribution_basic_scenario,1.919475,-0.281411
lift_lines_6km_contribution_basic_scenario,1.802826,0.778837
nearest_airport_distance_km_contribution_basic_scenario,1.652898,0.259612
avg_snowfall_cm_per_season_contribution_basic_scenario,1.528532,-0.186308
avg_freezing_days_per_season_contribution_basic_scenario,1.500668,-0.578857
lodging_objects_6km_contribution_basic_scenario,1.428493,0.673822
avg_stay_nights_2025_contribution_basic_scenario,1.181199,0.366325


Сильнее всего различают курорты в базовом сценарии параметр Поездки по региону в 2025 году `trips_2025`, на втором месте Рост поездок `trips_growth_2025` и на третьем месте количество лыжных трасс `piste_segments_6km`. Учитывая косвенность показателя по спросу, стоит собрать спрос более тщательно, например, брать спрос за период лыжного сезона (ориентировочно с ноября по март). Так можно немного сократить влияние других типов туризма, но не полностью.
Меньше всего влияют параметры количества объектов питания и развлечений `food_attraction_objects` и транспортные объекты `transport_objects`.

Если рассматривать с точки зрения корреляции, то наиболее сильная корреляция итогового рейтинга наблюдается с параметрами Количество подъемников `lift_lines_6km` (0,78), Поездки `trips_2025` (0,74), потом идут Количество трасс `piste_segments_6km` (0,69) и объекты размещения `lodging_objects` (0,67). 

Климатические показатели и цена отрицательно связаны с итоговым баллом. Это не значит, что снег снижает рейтинг: вклад снега всегда положительный. Курорты с большим количеством снега и морозных дней расположены в основном в регионах с невысоким спросом (Мурманская область, Сибирь, Дальний Восток). Курорты с наибольшим спросом — Подмосковье, Краснодарский край, Ленинградская область — теплее и по собранным данным получают меньше снега. Поскольку спрос весит вдвое больше климата, снежные курорты оказываются ниже несмотря на хороший климат. С ценой ситуация похожая: самые дорогие курорты (Роза Хутор, Красная Поляна) — одновременно самые популярные, поэтому низкий балл за цену у них сочетается с высоким местом. 

___
Проверим какие группы факторов повлияли на итоговый рейтиг.

In [17]:
# Создадим функцию для расчета влияния групп факторов на рейтинг
def get_param_group_contribution(contribution_scores, scoring_config, scenario_type):
    param_contributions = contribution_scores[['resort_id']].copy()
    for param_group in scoring_config['param_group'].unique():
        column_list = (scoring_config[scoring_config['param_group'] == param_group]['column'] + f'_contribution_{scenario_type}').tolist()
        param_contributions[param_group] = contribution_scores[column_list].sum(axis=1)
    return param_contributions

# Выгрузим данные по влиянию факторов на рейтинг
param_contributions_basic_scenario = get_param_group_contribution(contribution_scores, parameters_weights_by_scenario, 'basic_scenario')
parameter_contributions_basic_scenario = param_contributions_basic_scenario.merge(
    resort_scores[['resort_id', 'name_ru', 'rank_basic_scenario']],
    on='resort_id',
    how='left'
)
top_resort_param_contributions_basic_scenario = parameter_contributions_basic_scenario[parameter_contributions_basic_scenario['rank_basic_scenario'] <= 10]
display(top_resort_param_contributions_basic_scenario.sort_values('rank_basic_scenario').reset_index(drop=True))


,resort_id,Региональный спрос,Инфраструктура,Зимние условия,Доступность,Цена,name_ru,rank_basic_scenario
0,R030,37.66,16.36,12.78,4.89,2.50,Сорочаны,1
1,R001,32.36,23.60,5.62,4.46,0.00,Роза Хутор,2
2,R003,32.36,24.25,0.47,4.65,2.01,Газпром Поляна,3
3,R002,32.36,22.65,0.00,4.70,1.55,Красная Поляна,4
4,R029,27.32,14.41,11.50,3.39,2.50,Игора,5
5,R004,15.46,20.91,17.29,2.48,2.50,Шерегеш,6
6,R016,14.14,16.72,15.31,7.35,2.50,Горный воздух,7
7,R005,20.59,17.96,12.00,2.45,2.25,Архыз,8
8,R006,20.59,16.49,12.84,1.52,2.50,Домбай,9
9,R019,18.67,14.65,11.49,5.66,2.50,Бобровый лог,10


В базовом сценарии наибольший вес у регионального спроса, и по вкладу групп курорты топ-10 делятся на три типа.

**Высоко за счёт спроса**: «Сорочаны», «Роза Хутор», «Газпром Поляна», «Красная Поляна», «Игора». Региональный спрос даёт им 27–38 баллов, около половины итоговой оценки (у «Сорочан» 51%, у «Розы Хутор» 49%).

**Высоко за счёт климата и инфраструктуры**: «Шерегеш», «Горный воздух», «Бобровый лог». Спрос даёт им всего 14–19 баллов, а в топ-10 их поднимают зимние условия и инфраструктура. Самый яркий пример — «Шерегеш»: при спросе вдвое ниже, чем у курортов Краснодарского края, он на 6-м месте (спрос — около 26% его оценки).

**Сбалансированные**: «Архыз» и «Домбай». Спрос около 20 баллов при заметных вкладах инфраструктуры и зимних условий.

У «Горного воздуха» (7-е место) баллы почти равномерно распределены между спросом, инфраструктурой и зимними условиями, а вклад доступности у него наибольший среди курортов топ-10.

«Сорочаны» опережают «Розу Хутор» на 8,15 балла: +5,3 за спрос, +7,2 за зимние условия и +2,5 за цену при −7,2 за инфраструктуру. По инфраструктуре «Сорочаны» уступают всем трём курортам Краснодарского края. Часть отрыва (2,5 балла) дал пропуск в данных: цены для «Сорочан» нет, и они получили нейтральный балл, а «Роза Хутор» как самый дорогой курорт — 0. Но даже при максимальной цене отрыв сократился бы до 5,65 балла, и место бы не изменилось. Гораздо сильнее на порядок в топе влияют данные о снеге. 

У трёх курортов Краснодарского края спрос одинаковый (32,36 балла), поэтому места между ними определяют только инфраструктура и цена. По зимним условиям они сильно уступают «Сорочанам», но, как показала проверка данных, снежные показатели для них, вероятно, занижены, поэтому их реальное отставание по этой группе, вероятно, меньше, чем отрадают собранные данные.

Собственные зимние показатели «Сорочан» надёжны: они совпадают в обоих источниках. Но их преимущество над курортами Краснодарского края зависит от снежных данных этих курортов. С данными о снеге из справочника проекта (они совпадают с контрольной таблицей ТЗ, Приложение Б) курорты Краснодарского края занимают 1–3 места, а «Сорочаны» опускаются на 4-е. Таким образом, состав четвёрки лидеров не зависит от источника погоды, а порядок внутри неё зависит. Выбор между ними требует проверки снежных данных.

____
Проверим как меняется ТОП курортов при измении сценариев.



Базовый рейтинг — топ, ответы на вопросы 1–3 ТЗ: что сильнее всего влияет, где спрос расходится с климатом и инфраструктурой, высокий индекс при низкой уверенности.
Сценарии — вопрос 4: кто сдвигается сильнее всего и кто устойчив.
Выводы и рекомендации — кого добавлять первыми, почему, закрытые курорты исключены, ограничения.


14 Что нужно исследовать
После расчёта рейтинга нужен анализ, а не только сортировка таблицы. В отчёте ответьте на следующие вопросы.
•	Какие три показателя сильнее всего влияют на итоговый порядок? +
•	Какие курорты находятся высоко из-за регионального спроса, а какие - благодаря климату или инфраструктуре? +
•	У каких объектов высокий индекс сочетается с низкой уверенностью? +/-
•	Какие места меняются при увеличении каждого группового веса на 20 процентов?
•	Есть ли пары соседних курортов, для которых OSM считает одни и те же объекты? +/-
•	Как расширенные показатели Росстата меняют первые десять мест?
•	Каких данных не хватает, чтобы говорить о фактическом спросе на отдельный курорт?


Дашборд
Основной экран - таблица рейтинга с фильтром по региону, компонентами, confidence_score и флагами качества. Отдельно нужны профиль курорта, карта, сравнение двух курортов и страница качества данных. На каждом экране пользователь должен иметь возможность увидеть источник и дату обновления показателя.


In [18]:
# Выгрузим рейтинг курортов при изменении сценария
resort_scores_different_scenario = resort_scores[['name_ru', 'region', 'rank_basic_scenario',
                                                  'rank_climate_scenario', 'rank_access_scenario',
                                                  'basic_climat_scenario_diff', 'basic_access_scenario_diff']]

display(resort_scores_different_scenario.style
        .background_gradient(subset=['basic_climat_scenario_diff'], cmap='RdYlGn', vmin=-22, vmax=22)
        .background_gradient(subset=['basic_access_scenario_diff'], cmap='RdYlGn', vmin=-22, vmax=22))

,name_ru,region,rank_basic_scenario,rank_climate_scenario,rank_access_scenario,basic_climat_scenario_diff,basic_access_scenario_diff
0,Сорочаны,Московская область,1,1,1,0,0
1,Роза Хутор,Краснодарский край,2,9,2,-7,0
2,Газпром Поляна,Краснодарский край,3,21,3,-18,0
3,Красная Поляна,Краснодарский край,4,26,5,-22,-1
4,Игора,Ленинградская область,5,6,8,-1,-3
5,Шерегеш,Кемеровская область,6,2,9,4,-3
6,Горный воздух,Сахалинская область,7,4,4,3,3
7,Архыз,Карачаево-Черкесская Республика,8,12,11,-4,-3
8,Домбай,Карачаево-Черкесская Республика,9,10,14,-1,-5
9,Бобровый лог,Красноярский край,10,15,7,-5,3


При сравнении рейтинга в трёх сценариях наиболее устойчив курорт «Сорочаны»: он на первом месте во всех сценариях. «Игора» тоже стабильно держится в топ-10, её места меняются от 5-го до 8-го.

**Климатический сценарий**

Сильнее всего меняется положение курортов Краснодарского края: «Красная Поляна» опускается с 4-го места на 26-е (−22 позиции), «Газпром Поляна» — с 3-го на 21-е (−18 позиций). «Роза Хутор» теряет 7 позиций и опускается со 2-го места на 9-е, но остаётся в топ-10. Такая изменчивость, вероятно, связана с заниженными снежными показателями для этих курортов: со снегом справочника они в климатическом сценарии занимают 1–3 места.

«Домбай» опускается с 9-го места на 10-е, «Архыз» и «Бобровый лог» выпадают из топ-10 (12-е и 15-е места). Улучшают положение «Шерегеш» (с 6-го места на 2-е) и «Горный воздух» (с 7-го на 4-е). В топ-10 в этом сценарии появляются северные и уральские курорты: «Губаха» поднимается на 3-е место, «Большой Вудъявр» — на 5-е, «КолаСпортЛэнд» — на 7-е, «Эльбрус» — на 8-е. «КолаСпортЛэнд» закрыт, поэтому его рост на рекомендацию не влияет.

**Сценарий доступности**

В этом сценарии положение верхней части рейтинга гораздо устойчивее. В топ-10 остаются 8 курортов из 10, первые три места по-прежнему занимают «Сорочаны», «Роза Хутор» и «Газпром Поляна». Немного снижаются «Красная Поляна» (−1 позиция), «Игора» и «Шерегеш» (−3 позиции), но остаются в топ-10. Растут «Горный воздух» (с 7-го места на 4-е) и «Бобровый лог» (с 10-го на 7-е). «Архыз» и «Домбай» выпадают из топ-10 (11-е и 14-е места). Вместо них входят «Морозная гора» (с 22-го места на 6-е) и «Красная Глинка» (на 10-е): оба курорта расположены близко к аэропортам.

**Итог**
От весов доступности рейтинг почти не зависит, от климата — сильно. Но верхнюю часть климатического сценария двигает прежде всего качество снежных данных, а не сам вес климата, поэтому при измененнии погодных данных рейтинг может измениться.

**Курорты, которые остаются в топ-10 во всех трёх сценариях:**

- «Сорочаны»
- «Роза Хутор»
- «Игора»
- «Шерегеш»
- «Горный воздух»

«Газпром Поляна» и «Красная Поляна» выпадают из топ-10 только в климатическом сценарии, и это, вероятно, следствие заниженных снежных данных: со снегом справочника они остаются на 1–3 местах. Поэтому их тоже стоит считать устойчивыми лидерами, с оговоркой о данных.
___
Проверим, как меняется рейтинг, если по очереди увеличить вес каждой из пяти групп параметров на 20%, пропорционально снизив остальные группы так, чтобы сумма весов осталась равной 100.

In [19]:
# Выгрузим рейтинг курортов при изменении сценария
resorts_rank_difference_by_scenario = resort_scores[['name_ru', 'region', 'rank_basic_scenario', 'rank_demand_group_scenario',
                                                     'rank_infrastructure_group_scenario', 'rank_weather_group_scenario',
                                                     'rank_accessibility_group_scenario', 'rank_price_group_scenario']].copy()
for rank_scenario in ['rank_demand_group_scenario', 'rank_infrastructure_group_scenario',
                      'rank_weather_group_scenario','rank_accessibility_group_scenario', 'rank_price_group_scenario']:
    resorts_rank_difference_by_scenario[f'diff_basic_{rank_scenario}'] = resorts_rank_difference_by_scenario['rank_basic_scenario'] - resorts_rank_difference_by_scenario[rank_scenario]

columns_for_display = ['name_ru', 'rank_basic_scenario'] + [col for col in resorts_rank_difference_by_scenario.columns if col.startswith('diff_')]
display(resorts_rank_difference_by_scenario[columns_for_display].style
        .background_gradient(subset=[col for col in resorts_rank_difference_by_scenario.columns if col.startswith('diff_')], cmap='RdYlGn', vmin=-6, vmax=6))


,name_ru,rank_basic_scenario,diff_basic_rank_demand_group_scenario,diff_basic_rank_infrastructure_group_scenario,diff_basic_rank_weather_group_scenario,diff_basic_rank_accessibility_group_scenario,diff_basic_rank_price_group_scenario
0,Сорочаны,1,0,0,0,0,0
1,Роза Хутор,2,0,0,0,0,0
2,Газпром Поляна,3,0,0,0,0,0
3,Красная Поляна,4,0,0,-2,0,0
4,Игора,5,0,-1,0,0,0
5,Шерегеш,6,0,1,2,0,0
6,Горный воздух,7,-2,0,0,0,0
7,Архыз,8,1,0,0,0,0
8,Домбай,9,1,0,-1,0,0
9,Бобровый лог,10,0,0,-1,0,0


При изменении веса любой группы параметров на 20% рейтинг остаётся достаточно устойчивым. Топ-10 практически не меняется, а первые три места («Сорочаны», «Роза Хутор», «Газпром Поляна») остаются прежними во всех пяти сценариях. Остальные курорты топ-10 сдвигаются не больше чем на 2 позиции. Например, при усилении зимних условий «Красная Поляна» и «Шерегеш» меняются местами (с 4-го на 6-е и с 6-го на 4-е), а при усилении спроса «Горный воздух» теряет 2 позиции, и выше поднимаются «Архыз» и «Домбай».

Единственное изменение состава топ-10 — при усилении зимних условий: «Бобровый лог» опускается на 11-е место, а «Губаха» поднимается с 11-го на 9-е. Более заметные сдвиги происходят в нижней части рейтинга и зависят от сценария.

**Спрос**
Сильнее всего рейтинг меняется при усилении спроса, до +/-6 позиций, но за пределами топ-10. Спрос — самая тяжёлая группа (40 пунктов), поэтому +20% дают ей наибольшую прибавку в абсолютных числах: +8 пунктов против +1–5 у остальных групп. Спрос задаётся по региону, поэтому поднимаются курорты регионов с высоким спросом: «Мраткино» (с 21-го места на 15-е) и «Солнечная долина» (с 23-го на 17-е). Опускаются курорты Кабардино-Балкарии и Мурманской области: «Эльбрус» (с 15-го на 21-е), «Чегет» (с 19-го на 24-е), «Большой Вудъявр» (с 14-го на 19-е) и «КолаСпортЛэнд» (с 18-го на 22-е). «КолаСпортЛэнд» закрыт, поэтому его положение на рекомендацию не влияет.

**Инфраструктура**
 Сильнее всего опускается «Гора Белая» (−4 позиции, с 16-го места на 20-е), сильнее всего поднимается «Чегет» (+3). Остальные курорты сдвигаются не больше чем на 1–2 позиции.

**Доступность**
Сильнее всего поднимается «Морозная гора» (+5 позиций, с 22-го места на 17-е), опускаются «Гора Белая» (с 16-го на 19-е) и «Чегет» (с 19-го на 22-е), каждый на 3 позиции.

**Зимние условия**
Изменения небольшие: не больше 2 позиций вверх или вниз.

**Цена**
Рейтинг не меняется совсем. На это есть три причины: в группе один показатель, его вес невелик (5 пунктов, после увеличения — 6), и у 23 курортов из 32 цены нет, поэтому они получают одинаковый нейтральный балл.

**Итог**
При изменении любого группового веса на 20% состав топ-10 практически сохраняется, а первые три места не меняются. Рейтинг устойчив к ошибке в оценке любого отдельного веса. Наименее устойчивы курорты середины рейтинга: «Гора Белая» опускается в трёх сценариях из пяти, «Чегет» сдвигается на 3–5 позиций в разные стороны в зависимости от сценария.

## Выводы и рекомендации

### Популярные курорты
Топ-10 курортов по базовому сценарию:
1. «Сорочаны» (Московская область)
2. «Роза Хутор» (Краснодарский край)
3. «Газпром Поляна» (Краснодарский край)
4. «Красная Поляна» (Краснодарский край)
5. «Игора» (Ленинградская область)
6. «Шерегеш» (Кемеровская область)
7. «Горный воздух» (Сахалинская область)
8. «Архыз» (Карачаево-Черкесская Республика)
9. «Домбай» (Карачаево-Черкесская Республика)
10. «Бобровый лог» (Красноярский край)

При ограниченных ресурсах на подключение курортов в первую очередь имеет смысл добавлять те, которые остаются в топе при разных сценариях и имеют высокую уверенность в итоговой оценке - так можно охватить наиболее широкую аудиторию.

**Первая очередь**: «Сорочаны», «Роза Хутор», «Газпром Поляна», «Красная Поляна». Уверенность у них от 75 до 100. «Газпром Поляну» и «Красную Поляну» стоит добавлять при условии проверки данных о снеге: именно снежные показатели сильно опускают эти курорты в климатическом стресс-сценарии, и это, вероятно, вызвано занижением снега в модели ERA5 для горных районов. Со снегом из справочника проекта они остаются на 1–3 местах.

**Вторая очередь**: «Игора», «Шерегеш», «Горный воздух». Эти курорты занимают более низкие места в десятке, но остаются в ней при всех проверках: и в стресс-сценариях, и при увеличении веса любой группы на 20%. У «Игоры» и «Горного воздуха» низкая формальная уверенность из-за отсутствия цены и пометки справочника о координатах, но сверка подтвердила их координаты, а цена не может вывести их из топ-10. Перед подключением всех трёх курортов желательно уточнить цену ски-пасса.

**Третья очередь**: «Архыз», «Домбай», «Бобровый лог». Они в топ-10 по базовому сценарию, но немного выпадают из него хотя бы в одной проверке (11–15-е места): «Архыз» – в обоих стресс-сценариях, «Домбай» — в сценарии доступности, «Бобровый лог» – в климатическом сценарии и при усилении зимних условий на 20%. У «Бобрового лога» тоже формально низкая уверенность из-за пометки о координатах и отсутствия цены, но сверка подтвердила, что его координаты соответствуют справочнику.

Для четырёх курортов из топ-10 точка справочника смещена: у «Розы Хутор» на 3,9 км, у «Красной Поляны», «Домбая» и «Горного воздуха» — на 1–2 км. Перед добавлением в каталог желательно уточнить их оценку по точным координатам.

В списке есть два закрытых курорта — «КолаСпортЛэнд» и «Аджигардак». Их нет смысла добавлять в каталог независимо от места в рейтинге.

### Ограничения
При оценке добавления курортов в каталог стоит учитывать:
1. Объекты OSM, расстояние до аэропорта и погода рассчитаны по координатам справочника, а у 10 курортов точка смещена от склона больше чем на 3 км.
2. Региональный спрос взят за год и по всему региону, поэтому может включать спрос на другие виды отдыха (например, летний отдых на море в Краснодарском крае). Все курорты одного региона получают одинаковые показатели спроса.
3. У нескольких регионов были аномальные (больше чем на 25%) изменения спроса в 2025 году по сравнению с 2024 годом.. Влияние таких экстремальных изменений минимизировано.
4. Климатические данные собраны из Open-Meteo по модели ERA5 (в ТЗ рекомендована ERA5-Land, но она не отдаёт снегопад). ERA5 может занижать снежные показатели в горах, и это влияет на климатический сценарий.
5. У трёх курортов («Холдоми», «Морозная гора», «Гладенькая») в радиусе поиска не найдено ни трасс, ни подъёмников: их точки смещены на 11–17 км. Поэтому их места в рейтинге занижены (сейчас все занимают места в нижней трети (22–31-е)).
6. Цены на взрослый ски-пасс есть только у 9 из 32 курортов. Цена слабо влияет на итоговый рейтинг (не больше 2,5 балла), но для более корректной оценки цены стоит актуализировать.
7. Уверенность не влияет на итоговую оценку, а показывает качество данных. При этом она не учитывает результаты фактической сверки координат.
8. Величина сдвига весов в стресс-сценариях (+20 пунктов) — допущение: при другой величине места изменятся иначе.
9. Расширенный слой Росстата (раздел 13 ТЗ) в первой версии не использовался.

### Дополнительные рекомендации
Чтобы уточнить расчёт и снять часть ограничений, можно дополнительно собрать данные. Этот список отвечает и на вопрос ТЗ о том, каких данных не хватает, чтобы оценить фактический спрос на отдельный курорт.

1. Пересчитать рейтинг по проверенным координатам: погода и реальные объекты инфраструктуры вокруг склона.
2. Детальнее проработать региональный спрос: например, взять период лыжного сезона (ноябрь — март или апрель) из помесячных данных Росстата.
3. Оценить динамику спроса за несколько лет. Сейчас используется рост за один год, а резкие скачки могут искусственно завышать или занижать оценку.
4. Собрать актуальные цены ски-пассов, а также стоимость размещения (среднюю и разброс).
5. Расширить климатические показатели (ветер, солнечные дни, дожди, высота склонов) и выбрать модель погоды, которая точнее отражает условия в горах.
6. Расширить показатели инфраструктуры: пункты проката, детские зоны, услуги инструкторов, длина и разнообразие трасс, трассы для разного уровня подготовки, варианты проживания.
7. Добавить отзывы с картографических сервисов (количество и средний рейтинг). Это косвенная оценка спроса на конкретный курорт: где больше отзывов, туда, скорее всего, больше едут, а где выше оценка, там больше нравится.
8. Проверить поисковый спрос по названиям курортов в Яндекс Wordstat: это позволит оценить спрос на конкретный курорт, а не на весь регион.